# SPX calibration study — Legendre chaos

Run 100 independent fits using the effective settings copied from this
folder's current `Calibration.ipynb`. Every fit starts with a fresh market-fitted
Heston model and its own chaos projection, using plain general-basis projection.
Projection and training seeds vary across runs; comparison seeds remain fixed.

The study preserves its quiet progress settings: no per-iteration loss printing
or per-run plots, and one `tqdm` progress bar. Numerical results are checkpointed
after each calibration. Timing, IV errors, bid/ask coverage, and iteration counts
are analyzed after the study.

Use **Restart Kernel → Run All** to run or resume the updated configuration.
All quotes are used for fitting; independent MC draws do not constitute held-out
market observations.


SPX inputs and numerical results are not distributed. Supply your own data as described in the repository README.


## 1. Imports and data

Load SPX data and the shared study workflow. This notebook owns its calibration configuration.


In [ ]:
from pathlib import Path
import importlib
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Real data/Legendre"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.core import chaos as general_chaos_module
general_chaos_module = importlib.reload(general_chaos_module)
from wiener_chaos.pricing import heston as heston_module
heston_module = importlib.reload(heston_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
from wiener_chaos.experiments import calibration as timing_module
timing_module = importlib.reload(timing_module)
from wiener_chaos.experiments import calibration as study
study = importlib.reload(study)
print(f"Study workflow loaded: {study.__file__}")

DATA_PATH = ROOT / "Real data/real_data.pkl"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "SPX inputs are private and are not distributed. Supply your own quotes "
        "and run Real data/Data.ipynb first; see the README input format."
    )
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Calibration settings

`CONFIG` is a local snapshot of this folder's `Calibration.ipynb`; changing the
single-run notebook later does not silently change this study. The exceptions
are the preserved study printing settings and independent-run controls.

The model uses `P=3`, `d=2`, a 32,768-path pricing chunk,
and the same 640,000-path Heston projection budget and optimizer options.
Calibration uses `lr=0.0001`, up to **600** iterations, patience **60**,
`tol_error=2e-08`, and `limit_error_above=2e-06`. Both regularization
weights are zero, degree-2 moment control variates are active immediately, and
control-variate weights use all pricing samples. Checkpoints and stopping monitor
training price MSE, matching the single run (`check_validation_set_MC=None`).

Each run uses **2,800,000 training MC paths**, refreshed every
**50** iterations. Independent final evaluation also uses **2,800,000 MC paths**
and the same fixed evaluation seed as the single-run notebook. The validation
seed is retained in the protocol and is unused while validation is disabled.
The Legendre basis has 10 functions per Brownian driver. All ten maturities
use Monte Carlo, and Heston projection uses the supported plain estimator.

`STUDY_OPTIONS['n_calibrations']` remains **100**; per-run printing and plots
remain disabled. Runs use seeds `base_seed + run_index`, with a separate training
stream offset by 3,000,000. A run hitting the iteration cap is flagged for review.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 2
if torch.cuda.is_available() and not 0 <= CUDA_DEVICE_INDEX < torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

N_MATURITIES = len(data["maturities"])
CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": "tensor",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 3,
    # Legendre basis: ten time functions per Brownian driver.
    "basis": "legendre",
    "M": 10,
    "base_jitter": 1e-12,
    # Paths evaluated per chunk by the general-basis pricing kernel.
    "pricing_chunk_size": 32_768,
    "init_scale": 1e-3,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # General-basis projection currently supports the plain estimator.
        "projection_control": "none",
        # Antithetic projection is currently supported only for piecewise bases.
        "projection_antithetic": False,
        # Independent projection estimates averaged before normalization.
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 5_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 128,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 201,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 500,
            # Heston optimizer tolerance for objective improvement.
            "ftol": 1e-12,
            # Heston optimizer tolerance for the gradient.
            "gtol": 1e-9,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["MC"] * 10,
        # MC paths per calibration maturity; None for quadrature.
        "N": [400_000] * 4 + [200_000] * 6,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 10,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 10,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 600,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 60,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 2e-8,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 2e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # MC paths reserved to estimate control-variate weights.
        #"n_samples_cv": 10_000,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": None,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
# Use the same per-maturity path counts with independent evaluation draws.
CONFIG["comparison_N_per_mat"] = list(CONFIG["pricing_calibr"]["N"])
OUTPUT_ROOT = STUDY_DIR / "Statistics calibration"
STUDY_OPTIONS = {
    # Number of independent calibration runs.
    "n_calibrations": 100,
    # Continue a matching study from saved completed runs.
    "resume": True,
    # Keep the validation MC sample common across runs.
    "fixed_validation_seed": 4_000_042,
    # Keep the evaluation MC sample common across runs.
    "fixed_evaluation_seed": 5_000_042,
}
# Timing reference saved by a healthy individual calibration in this folder.
TIMING_REFERENCE = (OUTPUT_ROOT /
    "single" / "result.json")
TIMING_TOLERANCE = 0.30  # Allow 30% longer than the individual reference.
# Resume this saved study without repeating completed runs.
RESUME_STUDY_DIR = None
training_paths = sum(n or 0 for n in CONFIG["pricing_calibr"]["N"])
comparison_paths = sum(n or 0 for n in CONFIG["comparison_N_per_mat"])
print(f"Device: {DEVICE} | Calibrations: {STUDY_OPTIONS['n_calibrations']} | "
      f"Training MC paths: {training_paths:,} | Validation/evaluation MC paths: {comparison_paths:,}")
print(f"Anchor lambda: {CONFIG['calibration']['anchor_lambda']:.1e} | "
      f"Stopping tolerance: {CONFIG['calibration']['tol_error']:.1e} | "
      f"Initialization: per-run surface Heston + plain Legendre projection")
maturity_table = pd.DataFrame({
    "maturity": data["maturities"],
    "quotes": [len(row) for row in data["strikes"]],
    "rate": data["r_s"],
    "dividend_yield": data["q_s"],
    "pricing": CONFIG["pricing_calibr"]["method"],
    "training_N": CONFIG["pricing_calibr"]["N"],
    "validation_evaluation_N": CONFIG["comparison_N_per_mat"],
})
display(maturity_table)

# Reuse completed studies automatically; set False to run or resume calibrations.
SAVED_RESULTS_DIR = OUTPUT_ROOT / "study"
LOAD_SAVED_RESULTS = (study.resolve_study_directory(SAVED_RESULTS_DIR) / "results.json").is_file()


## 3. Study workflow

Model construction, fresh Heston initialization, resampling, independent evaluation,
and checkpointing are handled by
[calibration.py](../../wiener_chaos/experiments/calibration.py). This notebook owns its local
configuration and study controls. Each run receives a fresh model and parameter.


## 4. Run or load the study

Load the latest completed study automatically when available. Otherwise, run
or resume the configured calibrations. Set `LOAD_SAVED_RESULTS=False` explicitly
to run or resume calibrations. Completed runs are checkpointed, and a completed
study becomes the default for analysis and plotting automatically.


In [ ]:
if LOAD_SAVED_RESULTS:
    if SAVED_RESULTS_DIR is None:
        raise ValueError("Set SAVED_RESULTS_DIR to a completed study folder before loading.")
    records, OUTPUT_DIR, ANALYSIS_CONFIG = study.load_saved_results(SAVED_RESULTS_DIR)
    print(f"Loaded {len(records)} saved calibrations from {OUTPUT_DIR}")
else:
    records, OUTPUT_DIR = study.run_spx_study(
        data, CONFIG, DATA_PATH, output_root=OUTPUT_ROOT,
        timing_reference=TIMING_REFERENCE, timing_tolerance=TIMING_TOLERANCE,
        resume_directory=RESUME_STUDY_DIR,
        **STUDY_OPTIONS,
    )
    ANALYSIS_CONFIG = CONFIG


## 5. Fit quality and runtime

Summarize actual post-fit IV MAE, bid/ask coverage, numerical failures, and timing.
Initialization includes each run's Heston fit and chaos projection. Calibration
time includes the optimizer and independent final pricing inside `model.calibrate()`;
subsequent IV conversion, report generation, and plotting are outside that timer.
The reported total is initialization plus calibration.

The per-run table shows optimizer iterations and iteration-cap hits. A capped run
is not automatically considered converged. Invalid IVs remain counted and
invalidate that run's IV MAE.


In [ ]:
run_table = pd.DataFrame([study.summarize_spx_run(record, data) for record in records]).set_index("run")
quality_columns = ["iv_mae_bp", "within_bid_ask_pct", "loss_ins", "loss_oos",
                   "invalid_iv_quotes"]
fit_summary = run_table[quality_columns].agg(["count", "mean", "std", "min", "max"]).T
timing_summary = run_table[["initialization_s", "calibration_s", "total_s"]].agg(
    ["count", "mean", "std", "min", "max"]
).T
display(fit_summary)
display(timing_summary)
print(f"Sum of initialization times: {run_table['initialization_s'].sum():.2f}s | "
      f"Sum of calibration times: {run_table['calibration_s'].sum():.2f}s | "
      f"Combined: {run_table['total_s'].sum():.2f}s")
print(f"Completed calibrations: {len(records)} | "
      f"Runs with invalid IV quotes: {int((run_table['invalid_iv_quotes'] > 0).sum())}")
display(run_table[["calibration_iterations", "reached_iteration_cap", "calibration_s", "iv_mae_bp"]])
print(f"Runs reaching the iteration cap: {int(run_table['reached_iteration_cap'].eq(True).sum())}")


In [ ]:
figures = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
valid_mae = pd.to_numeric(run_table["iv_mae_bp"], errors="coerce").dropna()
if not valid_mae.empty:
    valid_mae.plot.hist(ax=axes[0], bins=min(20, len(records)))
else:
    axes[0].text(.5, .5, "No runs with valid IV MAE", ha="center", transform=axes[0].transAxes)
axes[0].set(xlabel="IV MAE (bp)", title="Fit quality across calibrations")
run_table[["initialization_s", "calibration_s", "total_s"]].plot(ax=axes[1])
axes[1].set(xlabel="Calibration run", ylabel="Seconds", title="Initialization and calibration time")
fig.tight_layout()
plt.show()
figures["study_summary"] = fig

# Display one representative smile grid after the study, not one per run.
valid = run_table["iv_mae_bp"].dropna()
if not valid.empty:
    best_run = int(valid.idxmin())
    figures["best_calibration"] = study.plot_spx_smiles(data, records[best_run - 1]["iv_model"])
    print(f"Best valid run: {best_run} | IV MAE: {valid.loc[best_run]:.4f} bp")

## 6. Saved results

The study folder contains `manifest.json` and aggregate `results.json`, with
one checkpoint per calibration in `runs/run_XXX/result.json`. Coefficient tensors
are saved in `coefficients/`; fit and timing tables in `tables/`; summary plots
in `figures/`. Additional coefficient diagnostics are grouped in `diagnostics/`.
The relative `study_latest.json` pointer selects the completed study for loading
and plotting automatically.


In [ ]:
study.save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": torch.tensor([r["initial_coefficients"] for r in records], dtype=torch.float32),
                  "calibrated_coefficients": torch.tensor([r["d_a"] for r in records], dtype=torch.float32)},
    tables={"calibrations": run_table, "fit_summary": fit_summary, "timing_summary": timing_summary},
    figures=figures,
)
print(f"Study results saved in: {OUTPUT_DIR}")
